# Bài học 12 - Giảm lịch sử trò chuyện với Agent Scratchpad

Sổ tay này trình bày cách quản lý ngữ cảnh trong các cuộc hội thoại dài sử dụng Microsoft Agent Framework. Khi cuộc trò chuyện kéo dài, số lượng token tăng lên — cuối cùng vượt quá cửa sổ ngữ cảnh của mô hình. Chúng ta giải quyết điều này bằng **mẫu tóm tắt ngữ cảnh** và **agent scratchpad** cho bộ nhớ bền vững.

## Những gì bạn sẽ học:
1. **Tại sao quản lý ngữ cảnh quan trọng**: Hiểu về giới hạn token và cửa sổ ngữ cảnh
2. **Agents nhận biết ngữ cảnh**: Xây dựng agents quản lý ngữ cảnh cuộc trò chuyện của chính họ
3. **Mẫu tóm tắt ngữ cảnh**: Sử dụng công cụ để cô đọng lịch sử trò chuyện
4. **Agent Scratchpad**: Bộ nhớ bền vững tồn tại sau khi giảm ngữ cảnh

## Yêu cầu trước:
- Cài đặt Azure OpenAI với biến môi trường được cấu hình
- Hiểu biết về các khái niệm cơ bản về agent từ các bài học trước


## Thiết lập


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## Tại Sao Quản Lý Ngữ Cảnh Lại Quan Trọng

Mỗi LLM đều có một **cửa sổ ngữ cảnh** hữu hạn — số lượng token tối đa mà nó có thể xử lý trong một yêu cầu duy nhất. Khi một cuộc trò chuyện đa lượt tiến triển:

- **Số lượng token tăng tuyến tính** với mỗi tin nhắn của người dùng và phản hồi của trợ lý.
- **Token trong prompt chiếm phần lớn chi phí** vì toàn bộ lịch sử được gửi lại mỗi lượt.
- Cuối cùng, cuộc trò chuyện sẽ **vượt quá cửa sổ ngữ cảnh** và mô hình hoặc cắt bớt, hoặc báo lỗi.

### Các Chiến Lược Quản Lý Ngữ Cảnh

| Chiến Lược | Cách Hoạt Động | Đánh Đổi |
|---|---|---|
| **Cắt bớt** | Loại bỏ các tin nhắn cũ nhất | Mất ngữ cảnh ban đầu |
| **Tóm tắt** | Cô đọng các tin nhắn cũ thành một tóm tắt | Mất một số chi tiết, nhưng giữ được điểm chính |
| **Sổ ghi chú / Bộ nhớ bên ngoài** | Lưu trữ các sự kiện quan trọng bên ngoài cuộc trò chuyện | Cần gọi công cụ, nhưng duy trì được thông tin dù có giảm bớt |

Trong sổ tay này, chúng ta kết hợp **tóm tắt** với một **công cụ sổ ghi chú** để đại diện có thể duy trì tính liên tục ngay cả khi lịch sử trò chuyện được cô đọng.


## Tạo một Đại lý Nhận biết Ngữ cảnh


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## Mô phỏng một cuộc trò chuyện dài

Hãy cùng trải qua một cuộc trò chuyện nhiều lượt để xem cách ngữ cảnh được tích lũy. Đại lý nên giữ lại các chi tiết chính (sở thích, ngân sách, ngày đi du lịch) qua các lượt và thể hiện tính liên tục.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

Hãy chú ý cách đại lý giữ ngữ cảnh từ những lượt trước — nó biết về Nhật Bản, sushi, đền chùa, nhiếp ảnh, ngân sách 3000 đô la, du lịch một mình và khoảng thời gian tháng Tư. Trong một cuộc trò chuyện ngắn, điều này hoạt động tốt, nhưng khi cuộc trò chuyện kéo dài, toàn bộ lịch sử sẽ tốn kém khi gửi lại.

Hãy tiếp tục cuộc trò chuyện với nhiều lượt hơn để xem sự tích lũy ngữ cảnh:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## Mẫu Tóm Tắt Ngữ Cảnh

Khi cuộc trò chuyện phát triển, chúng ta có thể sử dụng **công cụ tóm tắt** để cô đọng ngữ cảnh tích lũy thành một định dạng súc tích. Đại lý gọi công cụ này để ghi lại những ưu tiên chính nhằm đảm bảo rằng ngay cả khi các tin nhắn cũ hơn bị loại bỏ, thông tin cơ bản vẫn được giữ lại.

Mẫu này là viên gạch xây dựng cho việc giảm lịch sử phức tạp hơn:
1. Đại lý xác định các sự kiện chính từ cuộc trò chuyện
2. Nó gọi công cụ tóm tắt để lưu trữ chúng
3. Các tin nhắn cũ hơn có thể được loại bỏ an toàn vì bản tóm tắt đã ghi lại những điều quan trọng

Dưới đây chúng ta định nghĩa một công cụ `summarize_preferences` mà đại lý có thể gọi để ghi lại một bản tóm tắt ngắn gọn về những gì nó đã học được.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## Tóm tắt

Trong bài học này, bạn đã học cách quản lý ngữ cảnh trong các cuộc hội thoại đại lý chạy lâu dài bằng cách sử dụng Microsoft Agent Framework:

### Những Khái Niệm Chính
- **Cửa sổ ngữ cảnh là có giới hạn** — mỗi token trong lịch sử hội thoại đều tốn phí và được tính vào giới hạn.
- **Công cụ tóm tắt** cho phép đại lý cô đọng ngữ cảnh tích lũy thành các bản tóm tắt ngắn gọn, giảm sử dụng token trong khi vẫn giữ được thông tin cần thiết.
- **Bảng ghi chú của đại lý** cung cấp bộ nhớ ngoài bền vững tồn tại qua mọi rút gọn hội thoại.

### Những Gì Bạn Đã Xây Dựng
- Một **đại lý nhận biết ngữ cảnh** duy trì sự liên tục trong các cuộc hội thoại nhiều lượt
- Một **công cụ tóm tắt** (`summarize_preferences`) ghi lại các chi tiết quan trọng của người dùng theo định dạng cô đọng
- Một **cuộc hội thoại nhiều lượt** trình diễn việc giữ ngữ cảnh và xử lý thay đổi

### Ứng Dụng Trong Thế Giới Thực
- **Bot dịch vụ khách hàng**: Ghi nhớ sở thích qua các phiên hỗ trợ dài
- **Trợ lý cá nhân**: Theo dõi dự án đang chạy mà không cần giải thích lại ngữ cảnh
- **Gia sư giáo dục**: Duy trì tiến trình học tập của học sinh qua nhiều tương tác

### Các Bước Tiếp Theo
- Triển khai công cụ bảng ghi chú đầy đủ với khả năng lưu trữ theo tệp
- Thêm tự động rút gọn lịch sử sau khi tóm tắt
- Kết hợp với cơ sở dữ liệu vectơ để tìm kiếm bộ nhớ ngữ nghĩa
- Xây dựng các đại lý có thể tiếp tục cuộc trò chuyện sau nhiều ngày với đầy đủ ngữ cảnh


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Tuyên bố miễn trừ trách nhiệm**:
Tài liệu này đã được dịch bằng dịch vụ dịch thuật AI [Co-op Translator](https://github.com/Azure/co-op-translator). Mặc dù chúng tôi cố gắng đảm bảo độ chính xác, xin lưu ý rằng bản dịch tự động có thể chứa lỗi hoặc sai sót. Tài liệu gốc bằng ngôn ngữ gốc nên được coi là nguồn tin chính thức. Đối với thông tin quan trọng, nên sử dụng dịch vụ dịch thuật chuyên nghiệp bởi con người. Chúng tôi không chịu trách nhiệm về bất kỳ hiểu lầm hoặc giải thích sai nào phát sinh từ việc sử dụng bản dịch này.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
